# F1 Podium Predictor — Training

Binary XGBoost classifier for the probability a driver finishes on the podium (P≤3).
Thin orchestration: all logic lives in the tested `f1pred` package (see
`specs/003-ml-model`). Run top-to-bottom; it downloads FastF1 seasons (cached in
`.fastf1-cache/`), trains, evaluates against the grid-top-3 baseline, shows SHAP,
and uploads a versioned artifact + model card to S3.

Prereq: `pip install -e ".[dev,notebook]"` from `ml/`, plus AWS creds for the
upload step. FastF1's cold download is rate-limited (500 calls/h), so the first
run over a multi-season window may need to resume across hourly windows; cached
sessions are free on re-run.

In [1]:
# Run from the ml/ project root so the relative .fastf1-cache/ and artifacts/
# paths resolve the same way whether launched via nbconvert or from notebooks/.
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("cwd:", Path.cwd())

cwd: /Users/martinschweiger/Developer/personal/F1-project/ml


In [2]:
import logging

import fastf1

from f1pred.data import RACE_COLUMNS, fastf1_load_race, fastf1_rounds_for_year, load_seasons
from f1pred.pipeline import run_pipeline

logging.basicConfig(level=logging.INFO)
# Phase 6: 0.2.0 adds 6 quali + practice features (see specs/006). 0.1.0 stays
# the live fallback until the roll-out gate (§3.5) says 0.2.0 wins on the same
# test fold (Constitution IX — no `latest/`, every version is explicit).
MODEL_VERSION = "0.2.0"
INCUMBENT_VERSION = "0.1.0"
# Temporal split: train ≤ TRAIN_MAX_YEAR, validate on VAL_YEAR, test on TEST_YEAR.
# FIRST_YEAR bounds the window (practice data is only reliable from 2019; D-5).
FIRST_YEAR = 2022
TRAIN_MAX_YEAR, VAL_YEAR, TEST_YEAR = 2023, 2024, 2025
# Backfill checkpoint from scripts/backfill_practice.py (T6) — the 12-feature race
# frame (R + Q + FP1–FP3 per round). Loading it is offline + deterministic and
# avoids re-tripping FastF1's 500-calls/h cap on the now-triple session loads.
RACES_CSV = f"artifacts/races_{FIRST_YEAR}_{TEST_YEAR}.csv"

/Users/martinschweiger/Developer/personal/F1-project/ml/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load historical races

Prefer the backfill checkpoint (`RACES_CSV`, written by `scripts/backfill_practice.py`)
— it already holds the 12-feature frame and is offline + deterministic. If it's
missing, fall back to a live FastF1 pull (cached; missing sessions skipped + logged).

In [3]:
from pathlib import Path

import pandas as pd

if Path(RACES_CSV).exists():
    races = pd.read_csv(RACES_CSV).reindex(columns=RACE_COLUMNS)
    print(f"loaded backfill checkpoint {RACES_CSV}")
else:
    print(f"{RACES_CSV} missing — falling back to a live FastF1 pull")
    races = load_seasons(
        range(FIRST_YEAR, TEST_YEAR + 1),
        rounds_for_year=fastf1_rounds_for_year,
        load_race=fastf1_load_race,
    )
print(f"{len(races)} race-driver rows across {races['year'].nunique()} seasons")
races.head()

loaded backfill checkpoint artifacts/races_2022_2025.csv
1838 race-driver rows across 4 seasons


,year,round,driver,constructor,circuit,grid_position,quali_gap_to_pole_s,is_wet,quali_segment_reached,quali_grid_delta,quali_teammate_gap_s,practice_best_pace_gap_s,practice_long_run_pace_s,practice_laps_count,points,finish_position
0,2022,1,LEC,Ferrari,Bahrain Grand Prix,1.0,0.000,False,3.0,0.0,-0.129,0.087,NaN,58,26.0,1.0
1,2022,1,SAI,Ferrari,Bahrain Grand Prix,3.0,0.129,False,3.0,0.0,0.129,0.584,-0.2575,65,18.0,2.0
2,2022,1,HAM,Mercedes,Bahrain Grand Prix,5.0,0.490,False,3.0,0.0,-0.204,1.185,0.0000,55,15.0,3.0
3,2022,1,RUS,Mercedes,Bahrain Grand Prix,9.0,0.694,False,3.0,0.0,0.204,0.593,-0.2260,67,12.0,4.0
4,2022,1,MAG,Haas F1 Team,Bahrain Grand Prix,7.0,0.903,False,3.0,0.0,-0.537,1.247,-0.1185,59,10.0,5.0


## 2. Run the pipeline (features → split → train → eval)

Temporal split, leakage-safe pre-race features, XGBoost with class weights.

In [4]:
result = run_pipeline(
    races,
    train_max_year=TRAIN_MAX_YEAR,
    val_year=VAL_YEAR,
    test_year=TEST_YEAR,
    version=MODEL_VERSION,
    fastf1_version=fastf1.__version__,
    limitations=(
        "Imbalanced target (~15% podium); FastF1 gaps dropped, not imputed. "
        "Quali + practice signal added in 0.2.0 (12 features). Practice pace is "
        "FP2+FP3 only, so sprint weekends (no FP2/FP3, ~18 rounds in 2022–2025) "
        "fall back to the neutral fill (D-6); no fuel/tyre correction on long-run "
        "pace (D-8). Live grid penalties are unknown pre-race, so quali_grid_delta "
        "is 0 at inference time."
    ),
)
print(f"train rows: {result.n_train}  test rows: {result.n_test}")

train rows: 845  test rows: 467


## 3. Metrics vs. baseline ("podium = grid ≤ 3")

In [5]:
import pandas as pd

pd.DataFrame({"model": result.metrics, "baseline": result.baseline}).T[
    ["accuracy", "log_loss", "roc_auc"]
]

,accuracy,log_loss,roc_auc
model,0.873662,0.28266,0.938115
baseline,0.922912,1.065008,0.852215


In [6]:
from f1pred.evaluate import calibration_figure, confusion_figure
from f1pred.features import build_features
from f1pred.schema import FEATURE_NAMES
from f1pred.split import temporal_split
from f1pred.target import podium_label

# Rebuild the test fold to render the figures.
_feat = build_features(races)
_data = _feat.assign(
    podium=podium_label(races, position_col="finish_position").reindex(_feat.index)
)
_test = temporal_split(
    _data, train_max_year=TRAIN_MAX_YEAR, val_year=VAL_YEAR, test_year=TEST_YEAR
).test

confusion_figure(result.metrics)

<Figure size 320x300 with 1 Axes>

In [7]:
calibration_figure(result.model, _test[list(FEATURE_NAMES)], _test["podium"])

<Figure size 400x400 with 1 Axes>

## 3.5 Roll-out gate — 0.2.0 vs. 0.1.0 (AC-4)

The 12-feature candidate goes live **only** if it beats the 6-feature incumbent on
the same test fold at **both** ROC-AUC (higher) and log-loss (lower). If the gate
fails, keep 0.1.0 live and do **not** merge `feature/quali-practice-features`
(`FEATURE_NAMES` is a global contract — no runtime dual-version support).

In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, roc_auc_score
from xgboost import XGBClassifier

from f1pred.evaluate import rollout_gate

# Score the incumbent 0.1.0 (its own 6-feature artifact) on the SAME 0.2.0 test
# fold so the comparison is apples-to-apples (AC-4). evaluate() can't be reused —
# it slices the full 12-name FEATURE_NAMES contract, which the 0.1.0 model lacks.
incumbent = XGBClassifier()
incumbent.load_model(f"artifacts/{INCUMBENT_VERSION}/model.json")
inc_features = incumbent.get_booster().feature_names or list(FEATURE_NAMES)[:6]
inc_proba = incumbent.predict_proba(_test[inc_features])[:, 1]
inc_pred = (inc_proba >= 0.5).astype(int)
_eps = 1e-6
incumbent_metrics = {
    "accuracy": float(accuracy_score(_test["podium"], inc_pred)),
    "log_loss": float(
        log_loss(_test["podium"], np.clip(inc_proba, _eps, 1 - _eps), labels=[0, 1])
    ),
    "roc_auc": float(roc_auc_score(_test["podium"], inc_proba)),
    "confusion_matrix": confusion_matrix(_test["podium"], inc_pred, labels=[0, 1]).tolist(),
}

gate = rollout_gate(result.metrics, incumbent_metrics)
print("GATE:", "PASS — roll out 0.2.0" if gate["passes"] else "FAIL — keep 0.1.0 live")
print(gate["reason"])
pd.DataFrame(
    {
        f"candidate {MODEL_VERSION}": result.metrics,
        f"incumbent {INCUMBENT_VERSION}": incumbent_metrics,
    }
).T[["accuracy", "log_loss", "roc_auc"]]

GATE: PASS — roll out 0.2.0
candidate wins: ROC-AUC +0.0040, log-loss -0.0004


,accuracy,log_loss,roc_auc
candidate 0.2.0,0.873662,0.28266,0.938115
incumbent 0.1.0,0.865096,0.283088,0.934142


## 4. SHAP — global + a single prediction

In [9]:
from f1pred.explain import importance_figure, one_prediction_figure, one_prediction_shap

importance_figure(result.importance)

<Figure size 500x300 with 1 Axes>

In [10]:
one_prediction_figure(one_prediction_shap(result.model, _test[list(FEATURE_NAMES)]))

<Figure size 500x300 with 1 Axes>

## 5. Publish the artifact + model card + history to S3

Needs AWS creds. Writes `models/<version>/{model.json, model_card.md, history.csv}`.
`history.csv` is the precomputed rolling-feature frame the inference λ loads instead
of re-fetching FastF1 (only the upcoming quali is pulled live). Uploading to
`models/0.2.0/` does **not** make it live — the inference λ's `MODEL_VERSION`
controls that (T12). Only flip the λ if the gate (§3.5) passed.

In [11]:
import boto3

from f1pred.artifact import publish
from f1pred.layout import bucket_name

region = "eu-central-1"
account = boto3.client("sts").get_caller_identity()["Account"]
s3 = boto3.client("s3", region_name=region)

out = publish(
    result.model,
    result.card_text,
    version=MODEL_VERSION,
    s3_client=s3,
    bucket=bucket_name(account, region),
    history=races[RACE_COLUMNS],
)
print("published", out)
print(result.card_text)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


INFO:f1pred.artifact:uploaded model 0.2.0 to s3://f1-data-128663321407-eu-central-1/models/0.2.0/


published artifacts/0.2.0
# Model Card — Podium Predictor `0.2.0`

Binary XGBoost classifier: probability a driver finishes on the podium (P≤3).

## Data

- Source: FastF1 3.8.3.
- Seasons / split (temporal, no shuffle): train ≤2023, val 2024, test 2025.
- Rows: 845 train, 467 test.

## Features (pre-race only — no leakage)

- `driver_form` — mean|SHAP| 1.090
- `grid_position` — mean|SHAP| 0.913
- `quali_gap_to_pole_s` — mean|SHAP| 0.453
- `constructor_form` — mean|SHAP| 0.320
- `practice_best_pace_gap_s` — mean|SHAP| 0.158
- `practice_long_run_pace_s` — mean|SHAP| 0.126

## Metrics (test) vs. baseline "podium = grid ≤ 3"

| model | accuracy | log_loss | roc_auc |
| ----- | -------- | -------- | ------- |
| XGBoost | 0.874 | 0.283 | 0.938 |
| baseline | 0.923 | 1.065 | 0.852 |

## Limitations

Imbalanced target (~15% podium); FastF1 gaps dropped, not imputed. Quali + practice signal added in 0.2.0 (12 features). Practice pace is FP2+FP3 only, so sprint weekends (no FP2/FP3, ~18 rounds 